# CHI@Edge Lease and Container Diagnostic

This notebook directly exercises the documented `python-chi` lease and container path for a CHI@Edge device. It is intended for operator diagnostics and Papermill execution, not as a student workflow.

In [ ]:
from pathlib import Path

edge_rc_file = "ansible/app-cred-coachable-chi-edge-openrc-unrestricted.sh"
lease_name = "arm-01-notebook-lease"
lease_days = 7
device_name = "soarm101-1"
container_name = "arm-01-notebook-container"
image_ref = "rianders/lerobot-soarm101:chi-edge"
device_profiles = ["ttyacm0", "ttyacm1", "video0", "video1", "video2", "video3"]
ts_hostname = "arm-01-notebook"
restart_container = True
create_container_flag = True
wait_timeout = 600


In [ ]:
import contextlib
import io
import json
import os
import shlex
import subprocess
import time
from datetime import timedelta

import chi
from chi import lease as lease_api
from chi.container import create_container, get_container
from chi.lease import Lease
from dotenv import load_dotenv

repo_root = Path.cwd()
load_dotenv(repo_root / ".env", override=False)

def profile_list(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    stripped = str(value).strip()
    if stripped.lower() in ("none", "no", "false", "[]"):
        return []
    if stripped.startswith("["):
        parsed = json.loads(stripped)
        if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
            raise ValueError("device_profiles list syntax must contain only strings")
        return parsed
    return [item.strip() for item in stripped.split(",") if item.strip()]

device_profiles = profile_list(device_profiles)
print("normalized device_profiles:", device_profiles)

def load_openrc(path):
    path = repo_root / path if not Path(path).is_absolute() else Path(path)
    result = subprocess.run(
        ["bash", "-lc", f"set -a && source {shlex.quote(str(path))} && env"],
        capture_output=True,
        text=True,
        check=True,
    )
    for line in result.stdout.splitlines():
        if line.startswith("OS_") and "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    return path

def setup_chi_edge(path):
    path = load_openrc(path)
    if os.getenv("OS_REGION_NAME") != "CHI@Edge":
        raise RuntimeError(f"{path} is for {os.getenv('OS_REGION_NAME')}, not CHI@Edge")

    auth_type = os.getenv("OS_AUTH_TYPE", "").lower()
    has_app_credential = bool(os.getenv("OS_APPLICATION_CREDENTIAL_ID"))
    use_app_credential = auth_type == "v3applicationcredential" or has_app_credential
    if use_app_credential:
        for key in ("OS_PROJECT_ID", "OS_PROJECT_NAME", "OS_PROJECT_DOMAIN_ID", "OS_PROJECT_DOMAIN_NAME"):
            os.environ.pop(key, None)

    chi.reset()
    chi.use_site("CHI@Edge")
    if use_app_credential:
        chi.set("auth_type", "v3applicationcredential")
        chi.set("application_credential_id", os.environ.get("OS_APPLICATION_CREDENTIAL_ID"))
        chi.set("application_credential_secret", os.environ.get("OS_APPLICATION_CREDENTIAL_SECRET"))
    elif auth_type:
        chi.set("auth_type", os.getenv("OS_AUTH_TYPE"))

    return {
        "auth_type": os.getenv("OS_AUTH_TYPE"),
        "region": os.getenv("OS_REGION_NAME"),
        "project_id": os.getenv("OS_PROJECT_ID") or os.getenv("OS_PROJECT_NAME") or "app-credential-scoped",
        "uses_app_credential": use_app_credential,
    }

auth_summary = setup_chi_edge(edge_rc_file)
print(json.dumps(auth_summary, indent=2))


In [ ]:
def matching_active_lease(name):
    for item in lease_api.list_leases():
        if item.name == name and item.status in ("ACTIVE", "PENDING"):
            return item
    return None

lease = matching_active_lease(lease_name)
if lease:
    print(f"Reusing {lease.status} lease {lease.name}: {lease.id}")
else:
    print(f"Creating CHI@Edge device lease {lease_name!r} for {device_name!r}")
    lease = Lease(name=lease_name, duration=timedelta(days=lease_days))
    lease.add_device_reservation(device_name=device_name, amount=1)
    print("Device reservations:", lease.device_reservations)
    lease.submit(wait_for_active=True, wait_timeout=wait_timeout, idempotent=True)
    print(f"Lease {lease.name} reached {lease.status}: {lease.id}")

reservation_id = lease.device_reservations[0]["id"]
print("reservation_id:", reservation_id)


In [ ]:
if create_container_flag:
    existing = get_container(container_name)
    if existing and restart_container:
        print(f"Deleting existing container {container_name}: {existing.id} [{existing.status}]")
        existing.delete()
        for _ in range(60):
            time.sleep(2)
            if get_container(container_name) is None:
                break
        else:
            raise TimeoutError("Timed out waiting for existing container deletion")
    elif existing:
        print(f"Reusing existing container {container_name}: {existing.id} [{existing.status}]")

    container = get_container(container_name)
    if not container:
        env = {
            "LEADER_PORT": "/dev/ttyACM0",
            "FOLLOWER_PORT": "/dev/ttyACM1",
            "CAMERA_INDEX": "0",
            "TS_HOSTNAME": ts_hostname,
        }
        ssh_pub_path = Path.home() / ".ssh" / "id_ed25519.pub"
        if ssh_pub_path.exists():
            env["SSH_PUBKEY"] = ssh_pub_path.read_text().strip()
        for key in ("HF_TOKEN", "HF_USER", "TS_AUTHKEY"):
            if os.getenv(key):
                env[key] = os.getenv(key)

        print("Creating container with device_profiles:", device_profiles)
        created = create_container(
            name=container_name,
            image=image_ref,
            reservation_id=reservation_id,
            command=["sleep", "infinity"],
            environment=env,
            device_profiles=device_profiles,
            hints={"platform_version": "2"},
        )
        container = get_container(created.uuid)
        if not container:
            raise RuntimeError(f"Container create returned {created.uuid}, but get_container could not retrieve it")
        container.wait(status="Running", timeout=wait_timeout)

    current = container.zun_container
    print(json.dumps({
        "name": container.name,
        "id": container.id,
        "status": current.status,
        "status_reason": getattr(current, "status_reason", None),
        "addresses": getattr(current, "addresses", None),
    }, indent=2, default=str))
    if current.status == "Error":
        raise RuntimeError(getattr(current, "status_reason", "container entered Error"))
else:
    print("create_container_flag is False; skipping container creation")
